In [ ]:
import TechAna_DRAFT as TechAna
import pandas as pd
import requests
import numpy as np
import importlib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
INDUSTRY_MAP = {
    1: {'name': 'Banks','module': 'TotalScore_Bank'},
    2: {'name': 'Consumer','module': 'TotalScore_Consumer'},
    3: {'name': 'Financials','module': 'TotalScore_Financials'},
    4: {'name': 'Construction_and_materials','module': 'TotalScore_CM'},
    5: {'name': 'Goods_and_services','module': 'TotalScore_GS'},
    6: {'name': 'HealthCare','module': 'TotalScore_HealthCare'},
    7: {'name': 'Insurance','module': 'TotalScore_Insurance'},
    8: {'name': 'Materials','module': 'TotalScore_Materials'},
    9: {'name': 'RealEstate','module': 'TotalScore_RealEstate'},
    10: {'name': 'Utilities_and_Energy','module': 'TotalScore_UtiEne'},
    11: {'name': 'TechTele','module': 'TotalScore_TechTele'},
}

In [ ]:
def _parse_stock_payload(payload):
    records = []
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        for sym, rows in payload.items():
            if isinstance(rows, list):
                for r in rows:
                    if 'symbol' not in r:
                        r = {**r, 'symbol': sym}
                    records.append(r)
            elif isinstance(rows, dict):
                if 'symbol' not in rows:
                    rows = {**rows, 'symbol': sym}
                records.append(rows)

    df = pd.DataFrame(records)
    if df.empty:
        return df

    for col in ['date', 'Date', 'trading_date', 'TradingDate']:
        if col in df.columns:
            df['date'] = pd.to_datetime(df[col], errors='coerce')
            break

    # Use Adj Close for all OHLC if available
    if 'adj_close' in df.columns:
        adj = pd.to_numeric(df['adj_close'], errors='coerce')
        df['open'] = adj
        df['high'] = adj
        df['low'] = adj
        df['close'] = adj

    else:
        for col in ['open', 'Open']:
            if col in df.columns:
                df['open'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['high', 'High']:
            if col in df.columns:
                df['high'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['low', 'Low']:
            if col in df.columns:
                df['low'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['close', 'Close']:
            if col in df.columns:
                df['close'] = pd.to_numeric(df[col], errors='coerce')
                break

    return df.dropna(subset=['symbol', 'date'])


def _compute_max_drawdown(series):
    if series is None or len(series) == 0:
        return 0.0
    running_max = series.cummax()
    drawdown = (series - running_max) / running_max
    return float(drawdown.min()) if len(drawdown) > 0 else 0.0


def _fetch_prices(symbols, start_date, end_date):
    if not symbols:
        return pd.DataFrame()
    params = {
        "symbols": ",".join(symbols),
        "start_date": start_date,
        "end_date": end_date
    }
    resp = requests.get(
        "http://192.168.8.190:8000/MKD/stock_daily",
        params=params,
        headers={"accept": "application/json"},
        timeout=30
    )
    resp.raise_for_status()
    payload = resp.json()
    return _parse_stock_payload(payload)


def _run_quarter_trades(symbols, start_date, end_date, df_prices, entry_override=None, original_entry_override=None):
    entry_override = entry_override or {}
    original_entry_override = original_entry_override or {}

    trades = []
    for sym in symbols:
        df_sym = df_prices[df_prices['symbol'] == sym].sort_values('date').reset_index(drop=True)
        if df_sym.empty:
            continue

        entry_row = df_sym.iloc[0]
        entry_date = entry_row['date']
        entry_price = entry_override.get(sym, entry_row.get('open', np.nan))
        if pd.isna(entry_price):
            continue

        original_entry_price = original_entry_override.get(sym, entry_price)

        sl_price = entry_price * 0.85
        tp_price = entry_price * 1.25

        exit_date = df_sym.iloc[-1]['date']
        exit_price = df_sym.iloc[-1]['close'] if 'close' in df_sym.columns else entry_price
        exit_reason = 'Keep Position'

        start_idx = 3 if len(df_sym) > 3 else len(df_sym)
        for i in range(start_idx, len(df_sym)):
            row = df_sym.iloc[i]
            day_open = row.get('open', np.nan)
            day_low = row.get('low', np.nan)
            day_high = row.get('high', np.nan)

            # Stoploss check (-15%)
            if pd.notna(day_low) and day_low <= sl_price:
                if pd.notna(day_open) and day_open <= sl_price:
                    exit_price = day_open
                else:
                    exit_price = sl_price
                exit_date = row['date']
                exit_reason = 'Stop Loss'
                break

            # Take profit check (+25%)
            if pd.notna(day_high) and day_high >= tp_price:
                exit_price = tp_price
                exit_date = row['date']
                exit_reason = 'Take Profit'
                break

        ret_pct = (exit_price - entry_price) / entry_price if entry_price else 0.0
        cum_ret_pct = (exit_price - original_entry_price) / original_entry_price if original_entry_price else 0.0

        max_dd = 0.0
        max_ret = 0.0
        min_ret = 0.0
        if 'close' in df_sym.columns:
            hold_df = df_sym[(df_sym['date'] >= entry_date) & (df_sym['date'] <= exit_date)]
            close_series = hold_df['close'].dropna()
            price_path = pd.concat([pd.Series([entry_price]), close_series], ignore_index=True)
            max_dd = _compute_max_drawdown(price_path)

            if not close_series.empty and entry_price:
                max_ret = (close_series.max() - entry_price) / entry_price
                min_ret = (close_series.min() - entry_price) / entry_price

        trades.append({
            'Symbol': sym,
            'Entry_Date': entry_date,
            'Entry_Price': entry_price,
            'Original_Entry_Price': original_entry_price,
            'Exit_Date': exit_date,
            'Exit_Price': exit_price,
            'Exit_Reason': exit_reason,
            'Return_Pct': ret_pct,
            'Max_Return_Pct': max_ret,
            'Min_Return_Pct': min_ret,
            'Cum_Return_Pct': cum_ret_pct,
            'Max_Drawdown': max_dd
        })

    return pd.DataFrame(trades)

def print_summary(df_trades, label, start_date, end_date, industry_name):
    if df_trades.empty:
        print(f'No trades generated for {label}.')
        return

    win_rate = (df_trades['Return_Pct'] > 0).mean()
    avg_returns = df_trades['Return_Pct'].mean()
    max_drawdown = df_trades['Max_Drawdown'].min()
    max_return = df_trades['Return_Pct'].max()
    min_return = df_trades['Return_Pct'].min()

    summary_df = pd.DataFrame([{
        'Industry': industry_name,
        'Label': label,
        'Period_Start': start_date,
        'Period_End': end_date,
        'Win_Rate': win_rate,
        'Average_Return': avg_returns,
        'Max_Return': max_return,
        'Min_Return': min_return,
        'Max_Drawdown': max_drawdown,
        'Deals': len(df_trades)
    }])

    print(f'\nTrade Results ({label}):')
    print(df_trades.to_string(index=False))
    print('\nSummary:')
    print(summary_df.to_string(index=False))

def run_backtest(industry_id):
    if industry_id not in INDUSTRY_MAP:
        print(f"Error: Industry ID {industry_id} not found in configuration.")
        return

    config = INDUSTRY_MAP[industry_id]
    industry_name = config['name']
    module_name = config['module']
    
    print(f"STARTING BACKTEST FOR: {industry_id} - {industry_name} (Module: {module_name})")
    try:
        TotalScore_Module = importlib.import_module(module_name)
    except ImportError:
        print(f"Error: Could not import module '{module_name}'. Check if file exists.")
        return

    TECH_START_DATE = TechAna.START_DATE
    TECH_END_DATE = TechAna.END_DATE

    tech_start = pd.to_datetime(TECH_START_DATE)
    tech_end = pd.to_datetime(TECH_END_DATE)
    tech_q = tech_end.to_period('Q')

    curr_q = tech_q + 1
    next_q = curr_q + 1

    PREV_START_DATE = tech_q.start_time.strftime('%Y-%m-%d')
    PREV_END_DATE = tech_q.end_time.strftime('%Y-%m-%d')
    START_DATE = curr_q.start_time.strftime('%Y-%m-%d')
    END_DATE = curr_q.end_time.strftime('%Y-%m-%d')
    NEXT_START_DATE = next_q.start_time.strftime('%Y-%m-%d')
    NEXT_END_DATE = next_q.end_time.strftime('%Y-%m-%d')

# Ranking for quarter t-1 (entry list for backtest quarter t)
    df_total_prev = TotalScore_Module.get_total_score(PREV_START_DATE, PREV_END_DATE, industry=industry_name)
    df_rank_prev = df_total_prev[['Symbol', 'Final_Score']].copy()
    df_rank_prev = df_rank_prev.sort_values('Final_Score', ascending=False).reset_index(drop=True)
    TOP10_PREV = df_rank_prev.head(10)['Symbol'].tolist()

# Ranking for quarter t (used for rollover filter and next-quarter new entries)
    df_total_curr = TotalScore_Module.get_total_score(START_DATE, END_DATE, industry=industry_name)
    df_rank_curr = df_total_curr[['Symbol', 'Final_Score']].copy()
    df_rank_curr = df_rank_curr.sort_values('Final_Score', ascending=False).reset_index(drop=True)
    TOP10_CURR = df_rank_curr.head(10)['Symbol'].tolist()
    TOP13_CURR = df_rank_curr.head(13)['Symbol'].tolist()

    df_prices_curr = _fetch_prices(TOP10_PREV, START_DATE, END_DATE)
    df_trades_curr = _run_quarter_trades(TOP10_PREV, START_DATE, END_DATE, df_prices_curr)

    rollover_symbols = []
    entry_override_next = {}
    original_entry_override_next = {}

    if not df_trades_curr.empty:
        pending_df = df_trades_curr[df_trades_curr['Exit_Reason'] == 'Keep Position']
        for _, row in pending_df.iterrows():
            sym = row['Symbol']
            if sym in TOP13_CURR:
                rollover_symbols.append(sym)
                original_entry_price = row.get('Original_Entry_Price', row['Entry_Price'])
                if row.get('Return_Pct', 0) > 0:
                    reference_price = row['Exit_Price']
                else:
                    reference_price = original_entry_price

                entry_override_next[sym] = reference_price
                original_entry_override_next[sym] = original_entry_price
                df_trades_curr.loc[df_trades_curr['Symbol'] == sym, 'Exit_Reason'] = 'Rollover'

    print_summary(df_trades_curr, f"{curr_q.year}Q{curr_q.quarter}", START_DATE, END_DATE, industry_name)

    symbols_next = TOP10_CURR + [s for s in rollover_symbols if s not in TOP10_CURR]
    df_prices_next = _fetch_prices(symbols_next, NEXT_START_DATE, NEXT_END_DATE)
    df_trades_next = _run_quarter_trades(
        symbols_next,
        NEXT_START_DATE,
        NEXT_END_DATE,
        df_prices_next,
        entry_override=entry_override_next,
        original_entry_override=original_entry_override_next
    )

    print_summary(df_trades_next, f"{next_q.year}Q{next_q.quarter}", NEXT_START_DATE, NEXT_END_DATE, industry_name)

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 2 - Consumer (Module: TotalScore_Consumer)
Success
[Consumer] Mega Caps: ['MCH', 'VNM', 'MWG', 'MSN']
[Consumer] Large Caps: ['VJC', 'HVN', 'SAB', 'PNJ', 'FRT', 'HAG']
[Consumer] Mega Caps: ['MCH', 'VNM', 'MWG', 'MSN']
[Consumer] Large Caps: ['VJC', 'HVN', 'SAB', 'PNJ', 'FRT', 'HAG']

Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   DGW 2022-10-03     44284.07              44284.07 2022-11-09    36225.70     Stop Loss   -0.181970        0.148581       -0.181970       -0.181970     -0.287791
   TLG 2022-10-03     36291.52              36291.52 2022-11-11    30719.20     Stop Loss   -0.153543        0.139764       -0.153543       -0.153543     -0.257340
   PAN 2022-10-03     20023.75              20023.75 2022-10-24    16405.00     Stop Loss   -0.180723        0.021687       -0.180723       -0.180723     -0.198113
   MWG 2022

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 11 - TechTele (Module: TotalScore_TechTele)
Success
[TechTele] Mega Caps: ['VGI', 'FPT', 'FOX', 'CMG']
[TechTele] Large Caps: ['SGT', 'VEC', 'ICT', 'POT', 'UNI', 'ST8']
[TechTele] Mega Caps: ['VGI', 'FPT', 'FOX', 'CMG']
[TechTele] Large Caps: ['SGT', 'VEC', 'ICT', 'POT', 'UNI', 'ST8']

Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   CMG 2022-10-03     27310.86              27310.86 2022-11-04    21989.73   Stop Loss   -0.194836        0.044601       -0.194836       -0.194836     -0.229213
   ICT 2022-10-03     13637.97              13637.97 2022-11-24    11429.97   Stop Loss   -0.161901        0.000895       -0.161901       -0.161901     -0.162651
   SMT 2022-10-03     10190.08              10190.08 2022-11-15     8359.05   Stop Loss   -0.179688        0.046875       -0.179688       -0.179688     -0.216418
   FPT 2022-10-03 

In [ ]:
@"Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   VCB 2022-10-03     39712.50              39712.50 2022-12-30  45000.0000    Rollover    0.133144        0.203966       -0.121813        0.133144     -0.132867
   MBB 2022-10-03     10112.96              10112.96 2022-10-24   8436.4800   Stop Loss   -0.165775        0.026738       -0.165775       -0.165775     -0.187500
   VPB 2022-10-03     14978.70              14978.70 2022-12-30  15771.6900    Rollover    0.052941        0.088235       -0.138235        0.052941     -0.162857
   HDB 2022-10-03      9382.16               9382.16 2022-11-04   7852.4600   Stop Loss   -0.163043        0.005435       -0.163043       -0.163043     -0.167568
   TCB 2022-10-03     14290.10              14290.10 2022-10-11  11337.6000   Stop Loss   -0.206612        0.008264       -0.206612       -0.206612     -0.213115
   STB 2022-10-03     19200.00              19200.00 2022-10-11  15850.0000   Stop Loss   -0.174479        0.002604       -0.174479       -0.174479     -0.176623
   CTG 2022-10-03     13260.24              13260.24 2022-11-28  16575.3000 Take Profit    0.250000        0.263889       -0.083333        0.250000     -0.100000
   TPB 2022-10-03     11142.72              11142.72 2022-10-11   9040.3200   Stop Loss   -0.188679        0.006289       -0.188679       -0.188679     -0.193750
   MSB 2022-10-03      7860.80               7860.80 2022-10-11   6620.9000   Stop Loss   -0.157732        0.009376       -0.157732       -0.157732     -0.165555
   BID 2022-10-03     22886.37              22886.37 2022-11-25  28607.9625 Take Profit    0.250000        0.267829       -0.096672        0.250000     -0.103774

Summary:
Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
   Banks 2022Q4   2022-10-01 2022-12-31       0.4       -0.037024        0.25   -0.206612     -0.213115     10

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   CTG 2023-01-03     17189.20              17189.20 2023-03-31    17925.88 Keep Position    0.042857        0.110714       -0.021429        0.042857     -0.118971
   TCB 2023-01-03     12967.38              12967.38 2023-03-31    13392.54 Keep Position    0.032787        0.071038       -0.045537        0.032787     -0.108844
   VCB 2023-01-03     45000.00              39712.50 2023-03-31    51412.50 Keep Position    0.142500        0.200000        0.032500        0.294618     -0.113542
   STB 2023-01-03     23500.00              23500.00 2023-03-31    26200.00 Keep Position    0.114894        0.153191       -0.008511        0.114894     -0.138376
   BID 2023-01-03     29886.48              29886.48 2023-03-31    33513.48 Keep Position    0.121359        0.165049       -0.010922        0.121359     -0.068553
   TPB 2023-01-03     10231.68              10231.68 2023-03-31    11508.48 Keep Position    0.124789        0.155251        0.000000        0.124789     -0.080000
   MBB 2023-01-03      9734.40               9734.40 2023-03-31     9869.60 Keep Position    0.013889        0.094444       -0.047222        0.013889     -0.129442
   ACB 2023-01-03     13299.93              13299.93 2023-03-31    14647.50 Keep Position    0.101322        0.160793       -0.002202        0.101322     -0.094877
   SHB 2023-01-03      6485.44               6485.44 2023-03-31     6703.70 Keep Position    0.033654        0.076923       -0.059615        0.033654     -0.126785
   LPB 2023-01-03      8438.43               8438.43 2023-03-31     9520.28 Keep Position    0.128205        0.135531       -0.010989        0.128205     -0.096667
   VPB 2023-01-03     15771.69              14978.70 2023-03-31    18547.16 Keep Position    0.175978        0.187151       -0.067039        0.238236     -0.152284

Summary:
Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
   Banks 2023Q1   2023-01-01 2023-03-31       1.0        0.093839    0.175978    0.013889     -0.152284     11"@

SyntaxError: unterminated string literal (detected at line 1) (<ipython-input-6-307fe836e237>, line 1)

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 3 - Financials (Module: TotalScore_Financials)
Success
[Financials] Mega Caps: ['SSI', 'VND', 'VCI', 'HCM']
[Financials] Large Caps: ['MBS', 'FTS', 'BSI', 'CTS', 'TIN', 'DSC']
[Financials] Mega Caps: ['SSI', 'VND', 'VCI', 'HCM']
[Financials] Large Caps: ['MBS', 'FTS', 'BSI', 'CTS', 'TIN', 'DSC']

Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   OGC 2022-10-03     13700.00              13700.00 2022-10-11    11400.00   Stop Loss   -0.167883        0.000000       -0.167883       -0.167883     -0.167883
   VCI 2022-10-03     19486.88              19486.88 2022-10-26    16316.25   Stop Loss   -0.162706        0.040219       -0.162706       -0.162706     -0.195079
   FTS 2022-10-03     16738.30              16738.30 2022-10-26    13305.53   Stop Loss   -0.205085        0.057627       -0.205085       -0.205085     -0.248397
   CTS 

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 4 - Construction_and_materials (Module: TotalScore_CM)
[Construction_and_materials] Mega Caps: ['VGC', 'CC1', 'BMP', 'LGC']
[Construction_and_materials] Large Caps: ['VCG', 'CII', 'CTR', 'NTP', 'PC1', 'CTD']
[Construction_and_materials] Mega Caps: ['VGC', 'CC1', 'BMP', 'LGC']
[Construction_and_materials] Large Caps: ['VCG', 'CII', 'CTR', 'NTP', 'PC1', 'CTD']

Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   DPG 2022-10-03     20246.27              20246.27 2022-10-21    16396.83   Stop Loss   -0.190131        0.000000       -0.190131       -0.190131     -0.190131
   HUB 2022-10-03     14178.15              14178.15 2022-10-26    11771.78   Stop Loss   -0.169724        0.069388       -0.169724       -0.169724     -0.223597
   BMP 2022-10-03     38222.87              38222.87 2022-12-30    43878.00    Rollover    0.147951      

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 5 - Goods_and_services (Module: TotalScore_GS)
Success
[Goods_and_services] Mega Caps: ['ACV', 'MVN', 'GEE', 'VEA']
[Goods_and_services] Large Caps: ['GEX', 'GMD', 'VTP', 'PHP', 'PVT', 'HAH']
[Goods_and_services] Mega Caps: ['ACV', 'MVN', 'GEE', 'VEA']
[Goods_and_services] Large Caps: ['GEX', 'GMD', 'VTP', 'PHP', 'PVT', 'HAH']

Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   TCL 2022-10-03     27204.32              27204.32 2022-11-15    21647.34     Stop Loss   -0.204268        0.009146       -0.204268       -0.204268     -0.211480
   PVT 2022-10-03     12315.10              12315.10 2022-11-15     9784.60     Stop Loss   -0.205479        0.057534       -0.205479       -0.205479     -0.248705
   CLL 2022-10-03     21295.56              21295.56 2022-12-30    19914.34 Keep Position   -0.064860        0.017123       -0.1337

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 6 - HealthCare (Module: TotalScore_HealthCare)
['BIO', 'CNC', 'DBM', 'DHN', 'DPH', 'DPP', 'DTH', 'HDP', 'MRF', 'NDC', 'NDP', 'NTF', 'TW3', 'YTC']
Symbols with data: 31 / 45
Success
[HealthCare] Mega Caps: ['DHG', 'IMP', 'DHT', 'DVN']
[HealthCare] Large Caps: ['DBD', 'DCL', 'DTP', 'TRA', 'DMC', 'TNH']
[HealthCare] Mega Caps: ['DHG', 'IMP', 'DHT', 'DVN']
[HealthCare] Large Caps: ['DBD', 'DCL', 'DTP', 'TRA', 'DMC', 'TNH']

Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   DBD 2022-10-03     29527.81              29527.81 2022-12-30  28905.0200      Rollover   -0.021092        0.028536       -0.109181       -0.021092     -0.133896
   TRA 2022-10-03     81676.80              81676.80 2022-12-30  77412.2000      Rollover   -0.052213        0.030208       -0.083096       -0.052213     -0.109982
   VMD 2022-10-03     16929.00       

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 8 - Materials (Module: TotalScore_Materials)
Success
[Materials] Mega Caps: ['HPG', 'GVR', 'KSV', 'MSR']
[Materials] Large Caps: ['DGC', 'DCM', 'DPM', 'HSG', 'PHR', 'NKG']
[Materials] Mega Caps: ['HPG', 'GVR', 'KSV', 'MSR']
[Materials] Large Caps: ['DGC', 'DCM', 'DPM', 'HSG', 'PHR', 'NKG']

Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   DCM 2022-10-03     25628.63              25628.63 2022-11-14    21704.63   Stop Loss   -0.153110        0.057416       -0.153110       -0.153110     -0.199095
   DPM 2022-10-03     19201.44              19201.44 2022-11-15    15215.82   Stop Loss   -0.207569        0.123853       -0.207569       -0.207569     -0.294898
   HPG 2022-10-03     14948.78              14948.78 2022-10-24    12413.16   Stop Loss   -0.169621        0.000000       -0.169621       -0.169621     -0.169621
   VGS 2022-1

In [ ]:
@"Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   BIC 2022-10-03     13235.20              13235.20 2022-11-15    11187.20   Stop Loss   -0.154739        0.038685       -0.154739       -0.154739     -0.186220
   BMI 2022-10-03     17818.20              17818.20 2022-10-11    14090.40   Stop Loss   -0.209213        0.000000       -0.209213       -0.209213     -0.209213
   PVI 2022-10-03     34599.90              34599.90 2022-12-30    40507.20    Rollover    0.170732        0.170732       -0.126829        0.170732     -0.126829
   VNR 2022-10-03     14846.72              14846.72 2022-12-23    12526.92   Stop Loss   -0.156250        0.084821       -0.156250       -0.156250     -0.222222
   PRE 2022-10-03     12837.72              12837.72 2022-11-10    10698.10   Stop Loss   -0.166667        0.040230       -0.166667       -0.166667     -0.198895
   MIG 2022-10-03     13781.30              13781.30 2022-10-07    11655.26   Stop Loss   -0.154270        0.000000       -0.154270       -0.154270     -0.154270
   BVH 2022-10-03     43664.64              43664.64 2022-12-30    43589.64    Rollover   -0.001718        0.081830       -0.110442       -0.001718     -0.164151
   ABI 2022-10-03     17003.63              17003.63 2022-10-14    14390.60   Stop Loss   -0.153675        0.002227       -0.153675       -0.153675     -0.155556
   PTI 2022-10-03     28134.74              28134.74 2022-10-10    23801.19   Stop Loss   -0.154028        0.000000       -0.154028       -0.154028     -0.154028
   PGI 2022-10-03     23634.08              23634.08 2022-12-30    24334.40    Rollover    0.029632        0.060367       -0.125000        0.029632     -0.164912

Summary:
 Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
Insurance 2022Q4   2022-10-01 2022-12-31       0.2        -0.09502    0.170732   -0.209213     -0.222222     10

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   MIG 2023-01-03     11731.19              11731.19 2023-03-31    12148.80 Keep Position    0.035598        0.074433       -0.103560        0.035598     -0.165662
   PRE 2023-01-03     13414.42              13414.42 2023-03-31    13581.30 Keep Position    0.012440        0.132948       -0.040462        0.012440     -0.153061
   BVH 2023-01-03     43664.64              43664.64 2023-03-31    45320.13 Keep Position    0.037914        0.096825        0.021847        0.037914     -0.062500
   PVI 2023-01-03     40507.20              34599.90 2023-03-31    43882.80 Keep Position    0.083333        0.083333       -0.020833        0.268293     -0.078431
   BIC 2023-01-03     13824.00              13824.00 2023-03-31    14131.20 Keep Position    0.022222        0.055556       -0.062963        0.022222     -0.112281
   ABI 2023-01-03     15635.40              15635.40 2023-03-31    14955.60 Keep Position   -0.043478        0.011594       -0.107246       -0.043478     -0.117479
   PTI 2023-01-03     33268.33              33268.33 2023-01-31    27001.35     Stop Loss   -0.188377        0.000000       -0.188377       -0.188377     -0.188377
   PGI 2023-01-03     24334.40              23634.08 2023-03-31    24606.80 Keep Position    0.011194        0.026119       -0.029851        0.041158     -0.054545
   BMI 2023-01-03     15526.80              15526.80 2023-03-31    16142.40 Keep Position    0.039648        0.215859       -0.004405        0.039648     -0.144928
   VNR 2023-01-03     13123.44              13123.44 2023-03-31    15310.68 Keep Position    0.166667        0.222222        0.000000        0.166667     -0.070248

Summary:
 Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
Insurance 2023Q1   2023-01-01 2023-03-31       0.8        0.017716    0.166667   -0.188377     -0.188377     10"@

SyntaxError: unterminated string literal (detected at line 1) (<ipython-input-13-71ec61dbdb6e>, line 1)

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 10 - Utilities_and_Energy (Module: TotalScore_UtiEne)
Success
[Utilities_and_Energy] Mega Caps: ['GAS', 'BSR', 'PLX', 'POW']
[Utilities_and_Energy] Large Caps: ['REE', 'PGV', 'PVS', 'PVD', 'OIL', 'VSH']
[Utilities_and_Energy] Mega Caps: ['GAS', 'BSR', 'PLX', 'POW']
[Utilities_and_Energy] Large Caps: ['REE', 'PGV', 'PVS', 'PVD', 'OIL', 'VSH']

Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   CNG 2022-10-03     22436.70              22436.70 2022-10-24    18085.34     Stop Loss   -0.193939        0.006061       -0.193939       -0.193939     -0.198795
   REE 2022-10-03     46028.16              46028.16 2022-12-30    44959.20      Rollover   -0.023224        0.109290       -0.128415       -0.023224     -0.214286
   CHP 2022-10-03     17338.64              17338.64 2022-12-30    16922.88 Keep Position   -0.023979        0.05327

In [ ]:
@"Trade Results (2022Q4):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   NVL 2022-10-03     82000.00              82000.00 2022-11-02    69200.00     Stop Loss   -0.156098        0.000000       -0.156098       -0.156098     -0.156098
   BCM 2022-10-03     86769.00              86769.00 2022-11-04    71905.80     Stop Loss   -0.171296        0.002222       -0.171296       -0.171296     -0.173134
   FIR 2022-10-03     28850.41              28850.41 2022-12-30    29860.49      Rollover    0.035011        0.065646       -0.052516        0.035011     -0.077083
   DXG 2022-10-03     14656.95              14656.95 2022-10-11    11942.70     Stop Loss   -0.185185        0.000000       -0.185185       -0.185185     -0.185185
   VRE 2022-10-03     26100.00              26100.00 2022-12-30    26300.00      Rollover    0.007663        0.208812       -0.149425        0.007663     -0.221870
   PDR 2022-10-03     42641.10              42641.10 2022-11-01    35450.64     Stop Loss   -0.168627        0.005882       -0.168627       -0.168627     -0.173489
   NBB 2022-10-03     18050.00              18050.00 2022-10-25    15200.00     Stop Loss   -0.157895        0.135734       -0.157895       -0.157895     -0.258537
   KBC 2022-10-03     27000.00              27000.00 2022-10-11    21750.00     Stop Loss   -0.194444        0.000000       -0.194444       -0.194444     -0.194444
   NLG 2022-10-03     24801.90              24801.90 2022-10-11    20712.23     Stop Loss   -0.164893        0.015958       -0.164893       -0.164893     -0.178010
   VPI 2022-10-03     48448.10              48448.10 2022-12-30    43954.60 Keep Position   -0.092749        0.028668       -0.121417       -0.092749     -0.134551

Summary:
  Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
RealEstate 2022Q4   2022-10-01 2022-12-31       0.2       -0.124851    0.035011   -0.194444     -0.258537     10

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   NLG 2023-01-03     28400.36              28400.36 2023-01-12    23733.63     Stop Loss   -0.164319        0.000000       -0.164319       -0.164319     -0.164319
   KBC 2023-01-03     24700.00              24700.00 2023-03-31    24250.00 Keep Position   -0.018219        0.091093       -0.145749       -0.018219     -0.217069
   FIR 2023-01-03     29860.49              28850.41 2023-03-31    32286.54 Keep Position    0.081246        0.111704       -0.023256        0.119102     -0.057078
   VRE 2023-01-03     26300.00              26100.00 2023-03-31    29550.00 Keep Position    0.123574        0.152091       -0.020913        0.132184     -0.150165
   KDH 2023-01-03     21028.00              21028.00 2023-03-31    20727.60 Keep Position   -0.014286        0.010714       -0.130357       -0.014286     -0.139576
   VHM 2023-01-03     49400.00              49400.00 2023-02-24    41000.00     Stop Loss   -0.170040        0.078947       -0.170040       -0.170040     -0.230769
   BCM 2023-01-03     81622.80              81622.80 2023-03-31    80553.93 Keep Position   -0.013095        0.023810       -0.029762       -0.013095     -0.052326
   DXG 2023-01-03     10546.80              10546.80 2023-02-13     8375.40     Stop Loss   -0.205882        0.095588       -0.205882       -0.205882     -0.275168
   VIC 2023-01-03     28400.00              28400.00 2023-03-31    27500.00 Keep Position   -0.031690        0.042254       -0.075704       -0.031690     -0.113176
   TIP 2023-01-03     14064.16              14064.16 2023-03-31    13904.34 Keep Position   -0.011364        0.028409       -0.079545       -0.011364     -0.104972

Summary:
  Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
RealEstate 2023Q1   2023-01-01 2023-03-31       0.2       -0.042408    0.123574   -0.205882     -0.275168     10"@

SyntaxError: unterminated string literal (detected at line 1) (<ipython-input-15-7e5e0eec3a17>, line 1)